In [1]:
%uv pip install timm albumentations tqdm opencv-python-headless seaborn kaggle -q

import os, zipfile, torch
from pathlib import Path

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)
with open(kaggle_dir / 'kaggle.json', 'w') as f:
    f.write('{"username":"ahnafnafim","key":"YOUR_KEY"}')
os.chmod(kaggle_dir / 'kaggle.json', 0o600)

data_dir = Path('/tmp/data')
data_dir.mkdir(parents=True, exist_ok=True)
!kaggle datasets download -d araftahsanpavel/tgif-subset -p /tmp/data/ -q
with zipfile.ZipFile(data_dir / 'tgif-subset.zip', 'r') as z:
    z.extractall(data_dir)
(data_dir / 'tgif-subset.zip').unlink(missing_ok=True)

import torch.nn as nn
torch.backends.cudnn.enabled = True
x = torch.randn(2, 128, 48, 48).cuda()
out = nn.BatchNorm2d(128).cuda()(x)
print(f"✓ cuDNN works | GPU: {torch.cuda.get_device_name(0)}")
del x, out

Note: you may need to restart the kernel to use updated packages.
Dataset URL: https://www.kaggle.com/datasets/araftahsanpavel/tgif-subset
License(s): unknown
✓ cuDNN works | GPU: NVIDIA H100 80GB HBM3


In [2]:
from __future__ import annotations
import os, time, random, json
import numpy as np
from pathlib import Path
from typing import Any, Dict, List, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.amp import autocast, GradScaler

import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm.auto import tqdm
import timm

IMAGE_SIZE          = 384
EPOCHS              = 30
BATCH_SIZE          = 16
LEARNING_RATE       = 1e-4
WEIGHT_DECAY        = 1e-4
LR_PATIENCE         = 3
LR_FACTOR           = 0.5
LR_MIN              = 1e-6
EARLY_STOP_PATIENCE = 7
BINARY_THRESHOLD    = 0.5
GRAD_CLIP_NORM      = 1.0
MIXED_PRECISION     = True
SEED                = 42
NUM_WORKERS         = 4
PIN_MEMORY          = True
POS_WEIGHT          = 3.0

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
DATA_ROOT = Path('/tmp/data/subset')
SAVE_DIR  = Path('/root/full_novelty_augmented')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("=" * 70)
print("  Enhanced ForensicNet — FULL NOVELTY + AUGMENTATION")
print("=" * 70)
print(f"  GPU: {torch.cuda.get_device_name(0)}")
print(f"  SAVE_DIR: {SAVE_DIR}")
print("=" * 70)

  Enhanced ForensicNet — FULL NOVELTY + AUGMENTATION
  GPU: NVIDIA H100 80GB HBM3
  SAVE_DIR: /root/full_novelty_augmented


In [3]:
def infer_mask_path_from_fake(fake_path: Path, masks_root: Path) -> Optional[Path]:
    category = fake_path.parent.name
    name = fake_path.name
    if '_mask_' not in name:
        return None
    stem = name
    for tag in ['_sd2_', '_sdxl_', '_firefly_', '_dalle_', '_dalle2_']:
        if tag in stem:
            stem = stem.split(tag)[0] + '.png'
            break
    candidate = masks_root / category / stem
    if candidate.exists():
        return candidate
    mask_dir = masks_root / category
    if mask_dir.exists():
        for m in mask_dir.glob('*.png'):
            if name.startswith(m.name):
                return m
    return None


def build_index(split_dir: Path) -> List[Dict[str, Any]]:
    images_root = split_dir / 'images'
    masks_root  = split_dir / 'masks'
    fakes_root  = split_dir / 'fakes'
    samples = []
    if images_root.exists():
        for p in images_root.rglob('*_orig.png'):
            samples.append({'image_path': str(p), 'mask_path': None,
                           'is_fake': False, 'category': p.parent.name})
    if fakes_root.exists():
        for p in fakes_root.rglob('*.png'):
            if '_mask_' not in p.name:
                continue
            paired_mask = infer_mask_path_from_fake(p, masks_root)
            if paired_mask is None or not paired_mask.exists():
                continue
            samples.append({'image_path': str(p), 'mask_path': str(paired_mask),
                           'is_fake': True, 'category': p.parent.name})
    return samples


class TGIFDataset(Dataset):
    def __init__(self, samples, image_size, transform=None):
        self.samples = samples
        self.transform = transform or A.Compose([
            A.Resize(image_size, image_size, interpolation=cv2.INTER_LINEAR),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ])
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        img = cv2.cvtColor(cv2.imread(s['image_path'], cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        if s['mask_path'] is not None:
            mask = (cv2.imread(s['mask_path'], cv2.IMREAD_GRAYSCALE) >= 127).astype(np.uint8)
        else:
            mask = np.zeros((img.shape[0], img.shape[1]), dtype=np.uint8)
        out = self.transform(image=img, mask=mask)
        return {
            'image': out['image'].float(),
            'mask': out['mask'].float().unsqueeze(0),
            'is_fake': torch.tensor(s['is_fake'], dtype=torch.bool),
        }


print("Indexing ...")
train_samples = build_index(DATA_ROOT / 'training')
val_samples   = build_index(DATA_ROOT / 'validation')
test_samples  = build_index(DATA_ROOT / 'testing')

for name, samples in [('train', train_samples), ('val', val_samples), ('test', test_samples)]:
    n_fake = sum(1 for s in samples if s['is_fake'])
    print(f"  {name:<6}: {len(samples):>5} ({len(samples)-n_fake} real, {n_fake} fake)")

# Augmented training transforms
aug_tf = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.3),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.0, p=0.4),
    A.Resize(IMAGE_SIZE, IMAGE_SIZE, interpolation=cv2.INTER_LINEAR),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

val_tf = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE, interpolation=cv2.INTER_LINEAR),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

train_loader = DataLoader(TGIFDataset(train_samples, IMAGE_SIZE, aug_tf),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader = DataLoader(TGIFDataset(val_samples, IMAGE_SIZE, val_tf),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader = DataLoader(TGIFDataset(test_samples, IMAGE_SIZE, val_tf),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

print(f"  Train (aug): {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

Indexing ...
  train :  7559 (2100 real, 5459 fake)
  val   :  1023 (341 real, 682 fake)
  test  :  1029 (343 real, 686 fake)
  Train (aug): 473 | Val: 64 | Test: 65


In [4]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__(); self.smooth = smooth
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits).view(logits.size(0), -1)
        targets = targets.view(targets.size(0), -1)
        inter = (probs * targets).sum(dim=1)
        denom = probs.sum(dim=1) + targets.sum(dim=1)
        return 1.0 - ((2.0 * inter + self.smooth) / (denom + self.smooth)).mean()


class MetricAccumulator:
    def __init__(self):
        self.tp = self.fp = self.tn = self.fn = 0
    def update(self, preds_binary, targets):
        p = preds_binary.bool(); t = targets.bool()
        self.tp += int((p & t).sum().item())
        self.fp += int((p & ~t).sum().item())
        self.tn += int((~p & ~t).sum().item())
        self.fn += int((~p & t).sum().item())
    def compute(self):
        eps = 1e-8; tp, fp, tn, fn = self.tp, self.fp, self.tn, self.fn
        precision = tp / (tp + fp + eps)
        recall = tp / (tp + fn + eps)
        f1 = 2 * precision * recall / (precision + recall + eps)
        return {
            'pixel_accuracy': (tp + tn) / (tp + fp + tn + fn + eps),
            'precision': precision, 'recall': recall, 'f1': f1,
            'iou': tp / (tp + fp + fn + eps),
            'dice': 2 * tp / (2 * tp + fp + fn + eps),
            'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn,
        }


def _sobel_edges(mask):
    sobel_x = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32, device=mask.device).view(1,1,3,3)
    sobel_y = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=torch.float32, device=mask.device).view(1,1,3,3)
    ex = F.conv2d(mask.float(), sobel_x, padding=1)
    ey = F.conv2d(mask.float(), sobel_y, padding=1)
    return (torch.sqrt(ex**2 + ey**2 + 1e-8) > 0.5).float()


class FullNoveltyLoss(nn.Module):
    def __init__(self, pos_weight=POS_WEIGHT):
        super().__init__()
        self.register_buffer('pw', torch.tensor([pos_weight], dtype=torch.float32))
        self.dice = DiceLoss()
        self.w_seg = 1.0
        self.w_edge = 0.3
        self.w_deep = 0.2
        self.w_contrastive = 0.05
        self.temperature = 0.07

    def _seg_loss(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=self.pw.to(logits.device))
        return 0.5 * bce + 0.5 * self.dice(logits, targets)

    def _edge_loss(self, edge_logits, mask_targets):
        edge_gt = _sobel_edges(mask_targets)
        if edge_logits.shape[-2:] != edge_gt.shape[-2:]:
            edge_logits = F.interpolate(edge_logits, size=edge_gt.shape[-2:], mode='bilinear', align_corners=True)
        return F.binary_cross_entropy_with_logits(edge_logits, edge_gt)

    def _deep_loss(self, deep_preds, mask_targets):
        total = 0.0
        for pred, w in zip(deep_preds, [0.125, 0.25, 0.5]):
            target_ds = F.interpolate(mask_targets, size=pred.shape[-2:], mode='nearest')
            total = total + w * self._seg_loss(pred, target_ds)
        return total

    def _contrastive_loss(self, embeddings, labels):
        labels = labels.view(-1).float(); B = embeddings.shape[0]
        if B < 2:
            return torch.tensor(0.0, device=embeddings.device, requires_grad=True)
        sim = torch.matmul(embeddings, embeddings.T) / self.temperature
        sim = sim - sim.max(dim=1, keepdim=True).values.detach()
        labels_eq = labels.unsqueeze(0) == labels.unsqueeze(1)
        self_mask = ~torch.eye(B, dtype=torch.bool, device=embeddings.device)
        pos_mask = labels_eq & self_mask
        pos_counts = pos_mask.float().sum(dim=1)
        valid = pos_counts > 0
        if valid.sum() == 0:
            return torch.tensor(0.0, device=embeddings.device, requires_grad=True)
        exp_sim = torch.exp(sim) * self_mask.float()
        log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-8)
        loss = -(log_prob * pos_mask.float()).sum(dim=1) / (pos_counts + 1e-8)
        return loss[valid].mean()

    def forward(self, outputs, masks, labels=None):
        l_seg = self._seg_loss(outputs['mask'], masks)
        l_edge = self._edge_loss(outputs['edge'], masks)
        l_deep = self._deep_loss(outputs['deep'], masks)
        l_con = self._contrastive_loss(outputs['embed'], labels) if labels is not None else torch.tensor(0.0, device=masks.device)
        total = self.w_seg * l_seg + self.w_edge * l_edge + self.w_deep * l_deep + self.w_contrastive * l_con
        return {
            'total': total,
            'seg': l_seg.detach(), 'edge': l_edge.detach(),
            'deep': l_deep.detach(), 'contrastive': l_con.detach(),
        }

print("✓ OK")

✓ OK


In [5]:
class SRMConv(nn.Module):
    def __init__(self):
        super().__init__()
        f1 = np.array([[0,0,0,0,0],[0,-1,2,-1,0],[0,2,-4,2,0],[0,-1,2,-1,0],[0,0,0,0,0]], dtype=np.float32) / 4.0
        f2 = np.array([[-1,2,-2,2,-1],[2,-6,8,-6,2],[-2,8,-12,8,-2],[2,-6,8,-6,2],[-1,2,-2,2,-1]], dtype=np.float32) / 12.0
        f3 = np.array([[0,0,0,0,0],[0,0,0,0,0],[0,1,-2,1,0],[0,0,0,0,0],[0,0,0,0,0]], dtype=np.float32) / 2.0
        self.register_buffer('weight', torch.from_numpy(np.stack([f1, f2, f3])[:, None]))
    def forward(self, x):
        return torch.cat([F.conv2d(x[:, c:c+1], self.weight, padding=2) for c in range(3)], dim=1)

class BayarConv2d(nn.Module):
    def __init__(self, in_ch=3, out_ch=3, k=5):
        super().__init__(); self.k = k
        self.weight = nn.Parameter(torch.randn(out_ch, in_ch, k, k) * 0.01)
    def forward(self, x):
        w = self.weight.clone(); c = self.k // 2
        wn = w.clone(); wn[:, :, c, c] = 0.0
        w[:, :, c, c] = -wn.sum(dim=[2, 3])
        return F.conv2d(x, w, padding=c)

class CBAM(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__(); mid = max(ch // r, 8)
        self.ca = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(ch, mid), nn.ReLU(), nn.Linear(mid, ch), nn.Sigmoid())
        self.sa = nn.Sequential(nn.Conv2d(2, 1, 7, padding=3, bias=False), nn.Sigmoid())
    def forward(self, x):
        x = x * self.ca(x).view(x.size(0), -1, 1, 1)
        return x * self.sa(torch.cat([x.mean(1, keepdim=True), x.amax(1, keepdim=True)], 1))

class FPNBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = nn.Sequential(nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.GELU(), nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.GELU())
        self.cbam = CBAM(out_ch)
    def forward(self, x, skip=None):
        x = self.up(x)
        if skip is not None:
            if x.shape[2:] != skip.shape[2:]: x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)
            x = torch.cat([x, skip], dim=1)
        return self.cbam(self.conv(x))

class NoiseBranch(nn.Module):
    def __init__(self, out_ch=128):
        super().__init__()
        self.srm = SRMConv(); self.bayar = BayarConv2d(3, 3)
        self.enc = nn.Sequential(nn.Conv2d(12, 32, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(32), nn.GELU(), nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(64), nn.GELU(), nn.Conv2d(64, out_ch, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.GELU())
    def forward(self, x):
        return self.enc(torch.cat([self.srm(x), self.bayar(x)], dim=1))

class DCTStream(nn.Module):
    def __init__(self, out_channels=128):
        super().__init__()
        self.freq_conv = nn.Sequential(nn.Conv2d(3, 32, kernel_size=8, stride=8, padding=0, bias=False), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.Conv2d(32, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.Conv2d(64, out_channels, 1, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True))
    def forward(self, x):
        return self.freq_conv(x)

class SEFusionBlock(nn.Module):
    def __init__(self, in_ch, out_ch, reduction=16):
        super().__init__()
        self.proj = nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
        self.se = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(out_ch, max(out_ch // reduction, 4), bias=False), nn.ReLU(inplace=True), nn.Linear(max(out_ch // reduction, 4), out_ch, bias=False), nn.Sigmoid())
    def forward(self, x):
        x = self.proj(x); return x * self.se(x).view(x.shape[0], x.shape[1], 1, 1)

class ContrastiveHead(nn.Module):
    def __init__(self, in_ch, proj_dim=128):
        super().__init__()
        self.proj = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(in_ch, in_ch, bias=False), nn.ReLU(inplace=True), nn.Linear(in_ch, proj_dim, bias=False))
    def forward(self, x):
        return F.normalize(self.proj(x), dim=1)

class EnhancedForensicNet(nn.Module):
    """Full novelty: ConvNeXt-L + SRM + Bayar + DCT + SE + FPN/CBAM + Edge + DeepSup + Contrastive"""
    def __init__(self, image_size=IMAGE_SIZE, pretrained=True):
        super().__init__()
        self.image_size = image_size
        self.enc = timm.create_model('convnext_large', pretrained=pretrained, features_only=True, out_indices=(0,1,2,3))
        self.noise = NoiseBranch(out_ch=128)
        self.dct = DCTStream(out_channels=128)
        self.lat4 = SEFusionBlock(1536 + 128 + 128, 256)
        self.lat3 = nn.Conv2d(768, 256, 1); self.lat2 = nn.Conv2d(384, 128, 1); self.lat1 = nn.Conv2d(192, 64, 1)
        self.dec3 = FPNBlock(256, 256, 256); self.dec2 = FPNBlock(256, 128, 128)
        self.dec1 = FPNBlock(128, 64, 64); self.dec0 = FPNBlock(64, 0, 32)
        self.head = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True), nn.Conv2d(32, 16, 3, padding=1, bias=False), nn.BatchNorm2d(16), nn.GELU(), nn.Conv2d(16, 1, 1))
        self.edge_head = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True), nn.Conv2d(32, 16, 3, padding=1, bias=False), nn.BatchNorm2d(16), nn.GELU(), nn.Conv2d(16, 1, 1))
        self.deep_head3 = nn.Conv2d(256, 1, 1); self.deep_head2 = nn.Conv2d(128, 1, 1); self.deep_head1 = nn.Conv2d(64, 1, 1)
        self.contrast_head = ContrastiveHead(256, 128)

    def forward(self, x):
        s1, s2, s3, s4 = self.enc(x)
        n_feat = F.adaptive_avg_pool2d(self.noise(x), s4.shape[2:])
        dct_feat = F.adaptive_avg_pool2d(self.dct(x), s4.shape[2:])
        p4 = self.lat4(torch.cat([s4, n_feat, dct_feat], dim=1))
        d3 = self.dec3(p4, self.lat3(s3)); d2 = self.dec2(d3, self.lat2(s2))
        d1 = self.dec1(d2, self.lat1(s1)); d0 = self.dec0(d1)
        seg = self.head(d0)
        if seg.shape[2:] != (self.image_size, self.image_size):
            seg = F.interpolate(seg, size=(self.image_size, self.image_size), mode='bilinear', align_corners=False)
        edge = self.edge_head(d0)
        if edge.shape[2:] != (self.image_size, self.image_size):
            edge = F.interpolate(edge, size=(self.image_size, self.image_size), mode='bilinear', align_corners=False)
        return {
            'mask': seg, 'edge': edge,
            'deep': [self.deep_head3(d3), self.deep_head2(d2), self.deep_head1(d1)],
            'embed': self.contrast_head(p4),
        }

model = EnhancedForensicNet(pretrained=True).to(device)
total_params = sum(p.numel() for p in model.parameters())
model_size_mb = total_params * 4 / 1024**2
print(f"Params: {total_params/1e6:.1f}M | Size: {model_size_mb:.0f}MB")

with torch.no_grad():
    _t = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE).to(device)
    _o = model(_t)
    print("✓ Forward pass verified:")
    for k, v in _o.items():
        print(f"  {k}: {[x.shape for x in v] if isinstance(v, list) else v.shape}")
    del _t, _o; torch.cuda.empty_cache()

model.safetensors:   0%|          | 0.00/791M [00:00<?, ?B/s]

Params: 199.7M | Size: 762MB
✓ Forward pass verified:
  mask: torch.Size([2, 1, 384, 384])
  edge: torch.Size([2, 1, 384, 384])
  deep: [torch.Size([2, 1, 24, 24]), torch.Size([2, 1, 48, 48]), torch.Size([2, 1, 96, 96])]
  embed: torch.Size([2, 128])


In [6]:
criterion = FullNoveltyLoss(pos_weight=POS_WEIGHT).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=LR_FACTOR, patience=LR_PATIENCE, min_lr=LR_MIN)
scaler = GradScaler(enabled=MIXED_PRECISION)

best_val_iou = -1.0
best_epoch = 0
epochs_no_improve = 0
history = []
best_ckpt = SAVE_DIR / 'best_full_novelty_aug.pth'
latest_ckpt = SAVE_DIR / 'latest_full_novelty_aug.pth'

print(f"\nTraining Full Novelty + Augmentation for {EPOCHS} epochs ...\n")
overall_start = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()

    # Train
    model.train()
    train_loss, train_batches = 0.0, 0
    train_metrics = MetricAccumulator()
    for batch in tqdm(train_loader, desc=f'train {epoch}', leave=False):
        images = batch['image'].to(device, non_blocking=True)
        masks = batch['mask'].to(device, non_blocking=True)
        labels = batch['is_fake'].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda', enabled=MIXED_PRECISION):
            outputs = model(images)
            loss_dict = criterion(outputs, masks, labels)
            loss = loss_dict['total']
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()
        with torch.no_grad():
            train_metrics.update((torch.sigmoid(outputs['mask']) >= BINARY_THRESHOLD).float(), masks)
        train_loss += loss.item(); train_batches += 1
    train_loss /= max(train_batches, 1)
    train_m = train_metrics.compute()

    # Validate
    model.eval()
    val_loss, val_batches = 0.0, 0
    val_metrics_acc = MetricAccumulator()
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'val {epoch}', leave=False):
            images = batch['image'].to(device, non_blocking=True)
            masks = batch['mask'].to(device, non_blocking=True)
            labels = batch['is_fake'].to(device, non_blocking=True)
            with autocast('cuda', enabled=MIXED_PRECISION):
                outputs = model(images)
                loss_dict = criterion(outputs, masks, labels)
            val_metrics_acc.update((torch.sigmoid(outputs['mask']) >= BINARY_THRESHOLD).float(), masks)
            val_loss += loss_dict['total'].item(); val_batches += 1
    val_loss /= max(val_batches, 1)
    val_m = val_metrics_acc.compute()

    lr = optimizer.param_groups[0]['lr']
    epoch_secs = time.time() - epoch_start
    scheduler.step(val_m['iou'])

    history.append({
        'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
        'train_iou': train_m['iou'], 'val_iou': val_m['iou'],
        'train_f1': train_m['f1'], 'val_f1': val_m['f1'],
        'val_precision': val_m['precision'], 'val_recall': val_m['recall'],
        'lr': lr, 'epoch_seconds': epoch_secs})

    torch.save({
        'epoch': epoch, 'model': model.state_dict(),
        'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(),
        'best_val_iou': best_val_iou, 'history': history,
    }, latest_ckpt)

    if val_m['iou'] > best_val_iou:
        best_val_iou = val_m['iou']; best_epoch = epoch; epochs_no_improve = 0
        torch.save({
            'epoch': epoch, 'model': model.state_dict(),
            'best_val_iou': best_val_iou, 'history': history,
            'total_params': total_params, 'model_size_mb': model_size_mb,
        }, best_ckpt)
        print(f"  Epoch {epoch:>3}/{EPOCHS} | train loss {train_loss:.4f} | "
              f"val IoU {val_m['iou']:.4f} | val F1 {val_m['f1']:.4f} | "
              f"lr {lr:.2e} | {epoch_secs:.0f}s ✓ BEST")
    else:
        epochs_no_improve += 1
        if epoch % 5 == 0 or epochs_no_improve >= EARLY_STOP_PATIENCE - 1:
            print(f"  Epoch {epoch:>3}/{EPOCHS} | train loss {train_loss:.4f} | "
                  f"val IoU {val_m['iou']:.4f} | val F1 {val_m['f1']:.4f} | "
                  f"lr {lr:.2e} | {epoch_secs:.0f}s | patience {epochs_no_improve}/{EARLY_STOP_PATIENCE}")

    if epochs_no_improve >= EARLY_STOP_PATIENCE:
        print(f"\n  [early stop] at epoch {epoch}"); break

    with open(SAVE_DIR / 'training_progress.json', 'w') as f:
        json.dump(history, f, indent=2, default=str)

total_time = time.time() - overall_start
print(f"\nDone. Best IoU: {best_val_iou:.4f} at epoch {best_epoch} | Total: {total_time:.0f}s")


Training Full Novelty + Augmentation for 30 epochs ...



train 1:   0%|          | 0/473 [00:00<?, ?it/s]

val 1:   0%|          | 0/64 [00:00<?, ?it/s]

  Epoch   1/30 | train loss 1.2373 | val IoU 0.1371 | val F1 0.2412 | lr 1.00e-04 | 115s ✓ BEST


train 2:   0%|          | 0/473 [00:00<?, ?it/s]

...

val 29:   0%|          | 0/64 [00:00<?, ?it/s]

train 30:   0%|          | 0/473 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x2b591b3c8040>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1647, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x2b591b3c8040>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/site-packages/torch/utils/data/dataloader.py", l

val 30:   0%|          | 0/64 [00:00<?, ?it/s]

  Epoch  30/30 | train loss 0.2788 | val IoU 0.8603 | val F1 0.9249 | lr 2.50e-05 | 101s | patience 2/7

Done. Best IoU: 0.8654 at epoch 28 | Total: 4226s


In [7]:
ckpt = torch.load(best_ckpt, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model'])
print(f"Loaded epoch {ckpt['epoch']} (val IoU: {ckpt['best_val_iou']:.4f})")

model.eval()
test_metrics = MetricAccumulator()
with torch.no_grad():
    for batch in tqdm(test_loader, desc='test'):
        images = batch['image'].to(device, non_blocking=True)
        masks = batch['mask'].to(device, non_blocking=True)
        with autocast('cuda', enabled=MIXED_PRECISION):
            outputs = model(images)
        test_metrics.update((torch.sigmoid(outputs['mask']) >= BINARY_THRESHOLD).float(), masks)

test_m = test_metrics.compute()

# All comparison numbers
baseline_iou = 0.8483; baseline_f1 = 0.9179
novelty_noaug_iou = 0.0000  # FILL with your no-aug full novelty result
novelty_noaug_f1 = 0.0000   # FILL with your no-aug full novelty result

results_text = f"""
======================================================
   FINAL RESULT — Full Novelty + Augmentation [test]
======================================================
  Pixel Accuracy     : {test_m['pixel_accuracy']:.4f}
  Precision          : {test_m['precision']:.4f}
  Recall             : {test_m['recall']:.4f}
  F1 Score           : {test_m['f1']:.4f}
  IoU (Jaccard)      : {test_m['iou']:.4f}
  Dice Score         : {test_m['dice']:.4f}
  ─────────────────────────────────
  Model Size         : {model_size_mb:.1f} MB
  Total Params       : {total_params/1e6:.2f} M
  Total Train Time   : {total_time:.0f} s
  Best Epoch         : {best_epoch}
  ─────────────────────────────────
  COMPARISON:
  ForensicNet (baseline)    : IoU {baseline_iou:.4f} | F1 {baseline_f1:.4f}
  Full novelty (no aug)     : IoU {novelty_noaug_iou:.4f} | F1 {novelty_noaug_f1:.4f}
  Full novelty (+ aug)      : IoU {test_m['iou']:.4f} | F1 {test_m['f1']:.4f}
  Δ IoU (base → best)       : {test_m['iou'] - baseline_iou:+.4f}
======================================================
"""

print(results_text)
(SAVE_DIR / 'final_results.txt').write_text(results_text)

save_data = {
    'model': 'EnhancedForensicNet_FullNovelty_Augmented',
    'architecture': 'ConvNeXt-L + SRM + Bayar + DCT + SE + FPN/CBAM + Edge + DeepSup + Contrastive + Augmentation',
    'test_metrics': test_m,
    'history': history,
    'total_params': total_params,
    'model_size_mb': model_size_mb,
    'total_time': total_time,
    'best_epoch': best_epoch,
    'augmentation': True,
}
with open(SAVE_DIR / 'final_results.json', 'w') as f:
    json.dump(save_data, f, indent=2, default=str)

print(f"✓ Saved to {SAVE_DIR}/")
print(f"\n⚠ Download files before closing — storage is ephemeral!")

Loaded epoch 28 (val IoU: 0.8654)


test:   0%|          | 0/65 [00:00<?, ?it/s]


   FINAL RESULT — Full Novelty + Augmentation [test]
  Pixel Accuracy     : 0.9943
  Precision          : 0.9403
  Recall             : 0.9244
  F1 Score           : 0.9323
  IoU (Jaccard)      : 0.8732
  Dice Score         : 0.9323
  ─────────────────────────────────
  Model Size         : 761.9 MB
  Total Params       : 199.74 M
  Total Train Time   : 4226 s
  Best Epoch         : 28
  ─────────────────────────────────
  COMPARISON:
  ForensicNet (baseline)    : IoU 0.8483 | F1 0.9179
  Full novelty (no aug)     : IoU 0.0000 | F1 0.0000
  Full novelty (+ aug)      : IoU 0.8732 | F1 0.9323
  Δ IoU (base → best)       : +0.0249

✓ Saved to /root/full_novelty_augmented/

⚠ Download files before closing — storage is ephemeral!
